<a href="https://colab.research.google.com/github/Fisev/PZP-Project/blob/Fisev/Project.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Effective single threaded on CPU

In [13]:
import re
from collections import Counter
from collections import defaultdict
import time

!wget https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/data.txt -O data.txt
!wget https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/stop_words.txt -O stop_words.txt

with open("data.txt", 'r') as file:
    whole_text = file.read()

with open("stop_words.txt", 'r') as file:
    stop_words = file.read()

words = re.findall(r'\b[a-zA-Z]+\b', whole_text.lower())

print(f"regex words: {words}\n\n")

start_time = time.time()
splitted_stop_words = stop_words.split()
print(f"stop_words: {splitted_stop_words}")


filtered_words = [""] * 120000
word_counts = defaultdict(int)
total_words_sum = 0
max_key, max_value = "", 0
min_key, min_value = "", 0
i = 0
for word in words:
    if len(word) <= 8 and len(word) >= 4:
        if word not in splitted_stop_words:
            word_counts[word] += 1
            i += 1
            filtered_words[i] = word
            value = word_counts[word]

            if word_counts[word] > max_value:
                max_key, max_value = word, value

min_key, min_value = min(word_counts.items(), key=lambda item: item[1])
end_time = time.time()

print(f"Number of words: {i}")
print(f"The most frequent word is: '{max_key}' with {max_value} occurrences")
print(f"The least frequent word: '{min_key}' with {min_value} occurrences")
print(f"time elapsed: {end_time - start_time}")

--2024-11-24 21:12:48--  https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/data.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1257260 (1.2M) [text/plain]
Saving to: ‘data.txt’

data.txt            100%[===================>]   1.20M  --.-KB/s    in 0.04s   

2024-11-24 21:12:48 (30.3 MB/s) - ‘data.txt’ saved [1257260/1257260]

--2024-11-24 21:12:48--  https://raw.githubusercontent.com/Fisev/PZP-Project/refs/heads/main/stop_words.txt
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.111.133, 185.199.108.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.111.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 100 [text/plain]
Saving to: 

Not effective single threaded on CPU

In [12]:

with open("data.txt", 'r') as file:
    whole_text = file.read().lower()

with open("stop_words.txt", 'r') as file:
    stop_words = file.read().lower()

words = whole_text.split()
cleaned_words = [re.sub(r'[^a-zA-Z]', '', word) for word in words]

start_time = time.time()

stop_words = stop_words.split()
print(f"stop_words: {stop_words}")

cleaned_words = [word for word in cleaned_words if word not in stop_words]
print(f"cleaned words: {cleaned_words}")

cleaned_words1 = [word for word in cleaned_words if len(word) <= 8]
print(f"long words: {cleaned_words1}")

cleaned_words2 = [word for word in cleaned_words1 if len(word) >= 4]
print(f"short words: {cleaned_words2}")

cleaned_words3 = [word for word in cleaned_words2 if word not in stop_words]
print(f"filtered words: {cleaned_words3}")

word_counts = Counter(cleaned_words3)
most_common_word, occurrences_common_word = word_counts.most_common(1)[0]

end_time = time.time()
print(f"time elapsed: {end_time - start_time}")
print(f"most common word: '{most_common_word}' with occurrences {occurrences_common_word}")

least_common_word, occurrences_least_common_word = word_counts.most_common()[-1]
print(f"The least frequent word is '{least_common_word}' with {occurrences_least_common_word} occurrences.")

stop_words: ['version', 'gutenberg', 'warranty', 'electronic', 'thee', 'queequeg', 'barbarians', 'summer-house', 'ferrule', 'odorous']
cleaned words: ['the', 'project', 'ebook', 'of', 'moby', 'dick', 'or', 'the', 'whale', 'by', 'herman', 'melville', 'this', 'ebook', 'is', 'for', 'the', 'use', 'of', 'anyone', 'anywhere', 'at', 'no', 'cost', 'and', 'with', 'almost', 'no', 'restrictions', 'whatsoever', 'you', 'may', 'copy', 'it', 'give', 'it', 'away', 'or', 'reuse', 'it', 'under', 'the', 'terms', 'of', 'the', 'project', 'license', 'included', 'with', 'this', 'ebook', 'or', 'online', 'at', 'wwwgutenbergorg', 'title', 'moby', 'dick', 'or', 'the', 'whale', 'author', 'herman', 'melville', 'last', 'updated', 'january', '', '', 'posting', 'date', 'december', '', '', 'ebook', '', 'release', 'date', 'june', '', 'language', 'english', '', 'start', 'of', 'this', 'project', 'ebook', 'moby', 'dick', 'or', 'the', 'whale', '', 'produced', 'by', 'daniel', 'lazarus', 'and', 'jonesey', 'moby', 'dick', 'or

Multithreaded on CPU

In [34]:
import threading
import multiprocessing
import time

with open("data.txt", 'r') as file:
    whole_text = file.read().lower()

with open("stop_words.txt", 'r') as file:
    stop_words = file.read().lower()

def text_procesing(text_chunk, target_words):
    splitted_text_chunk = text_chunk.split()

    filtered_words = [""] * 60000
    word_counter = Counter()
    i = 0
    for word in splitted_text_chunk:
        if len(word) <= 8 and len(word) >= 4:
            if word not in target_words:
                word_counter[word] += 1
                i += 1
                filtered_words[i] = word

    return (word_counter, i)

start_time = time.time()

splitted_stop_words = stop_words.split()

num_chunks = cpu_core_count = multiprocessing.cpu_count()
chunk_size = len(whole_text) // num_chunks

print(f"Number of CPU cores available: {cpu_core_count}")

chunks = [""] * num_chunks
for i in range(0, num_chunks):
    j = 0
    if i != 0:
        while whole_text[i + j + chunk_size] != " ":
            j += 1

    chunks[i] = whole_text[i * chunk_size + j:i * chunk_size + chunk_size + j]

with multiprocessing.Pool(processes=multiprocessing.cpu_count()) as pool:
    results = pool.starmap(text_procesing, [(chunk, splitted_stop_words) for chunk in chunks])

combined_counts = Counter()
total_word_count = 0
for result in results:
    combined_counts += result[0]
    total_word_count += result[1]

end_time = time.time()

most_common_word, most_common_freq = combined_counts.most_common(1)[0]
least_common_word, least_common_freq = combined_counts.most_common()[-1]

print(f"Number of words: {total_word_count}")
print(f"The most frequent word is: '{most_common_word}' with {most_common_freq} occurrences")
print(f"The least frequent word: '{least_common_word}' with {least_common_freq} occurrences")
print(f"time elapsed: {end_time - start_time}")

Number of CPU cores available: 2
Number of words: 109303
The most frequent word is: 'that' with 2759 occurrences
The least frequent word: 'includes' with 1 occurrences
time elapsed: 0.4756636619567871
